# Лабораторная работа №5 — генеративные модели изображений

**Задание 1:** Neural Style Transfer (VGG19 + LBFGS)  
**Задание 2:** DCGAN на **EMNIST (digits)**, 32×32, 1 канал

**Kaggle:** Settings → GPU, **Internet ON** (VGG, EMNIST, torchmetrics).

**NST:** добавьте датасет с 6 картинками или задайте `INPUT_IMAGES_DIR` и имена в `PAIRS` — иначе создаются заглушки.

In [ ]:
import os
import sys

# Раскомментируйте при ImportError torchmetrics:
# import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchmetrics>=1.0"])

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets
from torchvision.utils import make_grid, save_image
from torch.nn.utils import spectral_norm
from PIL import Image

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

OUT = "/kaggle/working/lab5" if os.path.isdir("/kaggle/working") else "lab5_output"
os.makedirs(f"{OUT}/nst", exist_ok=True)
os.makedirs(f"{OUT}/gan", exist_ok=True)

# Путь к папке с картинками для NST (после Add Data на Kaggle)
INPUT_IMAGES_DIR = "/kaggle/input/nst-images"
if not os.path.isdir(INPUT_IMAGES_DIR):
    INPUT_IMAGES_DIR = os.path.join(OUT, "nst_upload")
os.makedirs(INPUT_IMAGES_DIR, exist_ok=True)
print("NST images:", INPUT_IMAGES_DIR)

## Часть 1 — Neural Style Transfer

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]


def load_image(path, size=256):
    image = Image.open(path).convert("RGB")
    t = transforms.Compose(
        [
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ]
    )
    return t(image).unsqueeze(0)


def denormalize_nst(tensor):
    mean = torch.tensor(imagenet_mean).view(3, 1, 1)
    std = torch.tensor(imagenet_std).view(3, 1, 1)
    img = tensor.clone().squeeze(0).cpu()
    img = img * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


def to_rgb01(t):
    mean = torch.tensor(imagenet_mean, device=t.device).view(1, 3, 1, 1)
    std = torch.tensor(imagenet_std, device=t.device).view(1, 3, 1, 1)
    x = (t * std + mean).clamp(0, 1)
    return x.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()


print("Загрузка VGG19...")
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device)
for p in vgg.parameters():
    p.requires_grad = False
vgg.eval()

style_layers = {"conv1_1": 0, "conv2_1": 5, "conv3_1": 10, "conv4_1": 19, "conv5_1": 28}
content_layers = {"conv4_2": 21}
all_layers = {**style_layers, **content_layers}

style_weights = {"conv1_1": 1.0, "conv2_1": 0.8, "conv3_1": 0.5, "conv4_1": 0.3, "conv5_1": 0.1}


def get_features(image, model, layers):
    idx_to_name = {v: k for k, v in layers.items()}
    max_idx = max(idx_to_name.keys())
    feats = {}
    x = image
    for i, layer in enumerate(model):
        x = layer(x)
        if i in idx_to_name:
            feats[idx_to_name[i]] = x
        if i == max_idx:
            break
    return feats


def content_loss(gf, cf):
    return F.mse_loss(gf, cf)


def gram_matrix(f):
    b, c, h, w = f.shape
    Fm = f.view(b, c, h * w)
    G = torch.bmm(Fm, Fm.transpose(1, 2))
    return G / (c * h * w)


def style_loss(gf, sf):
    return F.mse_loss(gram_matrix(gf), gram_matrix(sf))

In [ ]:
def run_nst(
    content_img,
    style_img,
    *,
    num_steps=300,
    alpha=1.0,
    beta=1e5,
    init_noise=False,
    log_every=50,
    snapshot_steps=None,
):
    if init_noise:
        generated = torch.randn_like(content_img).requires_grad_(True)
    else:
        generated = content_img.clone().requires_grad_(True)

    with torch.no_grad():
        cf = get_features(content_img, vgg, all_layers)
        sf = get_features(style_img, vgg, all_layers)

    optimizer = optim.LBFGS([generated], lr=1.0, max_iter=20)
    mean = torch.tensor(imagenet_mean, device=device).view(1, 3, 1, 1)
    std = torch.tensor(imagenet_std, device=device).view(1, 3, 1, 1)

    def closure():
        optimizer.zero_grad()
        gf = get_features(generated, vgg, all_layers)
        c_loss = content_loss(gf["conv4_2"], cf["conv4_2"])
        s_loss = 0.0
        for layer in style_layers:
            s_loss = s_loss + style_weights[layer] * style_loss(gf[layer], sf[layer])
        total = alpha * c_loss + beta * s_loss
        total.backward()
        closure.last = (total, c_loss, s_loss)
        return total

    snapshots = []
    for step in range(num_steps):
        optimizer.step(closure)
        with torch.no_grad():
            generated.data = (generated.data * std + mean).clamp(0, 1)
            generated.data = (generated.data - mean) / std
        if step % log_every == 0 or step == num_steps - 1:
            tl, cl, sl = closure.last
            print(f"  step {step}: total={tl.item():.4f} c={cl.item():.4f} s={sl.item():.4f}")
        if snapshot_steps is not None and step in snapshot_steps:
            snapshots.append((step, to_rgb01(generated)))

    return to_rgb01(generated), snapshots

In [ ]:
PAIRS = [
    ("pair1_content.jpg", "pair1_style.jpg"),
    ("pair2_content.jpg", "pair2_style.jpg"),
    ("pair3_content.jpg", "pair3_style.jpg"),
]


def resolve_path(fn):
    return os.path.join(INPUT_IMAGES_DIR, fn)


def ensure_placeholder_images():
    for pc, ps in PAIRS:
        cp, sp = resolve_path(pc), resolve_path(ps)
        if not os.path.isfile(cp):
            a = np.zeros((256, 256, 3), np.uint8)
            a[48:208, 48:208] = [90, 140, 210]
            Image.fromarray(a).save(cp)
        if not os.path.isfile(sp):
            b = np.zeros((256, 256, 3), np.uint8)
            for i in range(0, 256, 32):
                for j in range(0, 256, 32):
                    b[i : i + 32, j : j + 32] = (
                        [230, 60, 60] if (i // 32 + j // 32) % 2 == 0 else [50, 160, 90]
                    )
            Image.fromarray(b).save(sp)
    print("Проверьте картинки в", INPUT_IMAGES_DIR, "(заглушки, если файлов не было)")


ensure_placeholder_images()
IMG_SIZE = 256

In [ ]:
# Три пары: Content | Style | Result
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
beta0 = 1e5

for row, (pc, ps) in enumerate(PAIRS):
    c_t = load_image(resolve_path(pc), IMG_SIZE).to(device)
    s_t = load_image(resolve_path(ps), IMG_SIZE).to(device)

    axes[row, 0].imshow(Image.open(resolve_path(pc)).resize((IMG_SIZE, IMG_SIZE)))
    axes[row, 0].set_title("Content")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(Image.open(resolve_path(ps)).resize((IMG_SIZE, IMG_SIZE)))
    axes[row, 1].set_title("Style")
    axes[row, 1].axis("off")

    print(f"NST пара {row + 1}...")
    out, _ = run_nst(c_t, s_t, num_steps=300, alpha=1.0, beta=beta0, init_noise=False, log_every=100)
    axes[row, 2].imshow(out)
    axes[row, 2].set_title("Result")
    axes[row, 2].axis("off")
    Image.fromarray((np.clip(out, 0, 1) * 255).astype(np.uint8)).save(f"{OUT}/nst/pair{row + 1}_result.png")

plt.tight_layout()
plt.savefig(f"{OUT}/nst/three_pairs.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Эволюция оптимизации (шаги 0 … 299) — первая пара
pc, ps = PAIRS[0]
c_t = load_image(resolve_path(pc), IMG_SIZE).to(device)
s_t = load_image(resolve_path(ps), IMG_SIZE).to(device)
snaps = [0, 50, 100, 150, 200, 250, 299]

print("Эволюция NST...")
_, shots = run_nst(
    c_t, s_t, num_steps=300, alpha=1.0, beta=1e5, init_noise=False, log_every=100, snapshot_steps=snaps
)

fig, axes = plt.subplots(1, len(shots), figsize=(3 * len(shots), 3))
for ax, (st, rgb) in zip(np.atleast_1d(axes), shots):
    ax.imshow(rgb)
    ax.set_title(f"step {st}")
    ax.axis("off")
plt.suptitle("NST: эволюция")
plt.tight_layout()
plt.savefig(f"{OUT}/nst/evolution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Влияние β (одна пара)
betas = [1e3, 1e5, 1e7]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, beta in zip(axes, betas):
    print(f"beta={beta}")
    out, _ = run_nst(c_t, s_t, num_steps=300, alpha=1.0, beta=beta, init_noise=False, log_every=150)
    ax.imshow(out)
    ax.set_title(f"β = {beta:.0e}")
    ax.axis("off")
plt.suptitle("Баланс содержание / стиль")
plt.tight_layout()
plt.savefig(f"{OUT}/nst/beta_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Инициализация: content vs шум
print("Сравнение инициализации...")
out_c, _ = run_nst(c_t, s_t, num_steps=300, alpha=1.0, beta=1e5, init_noise=False, log_every=150)
out_n, _ = run_nst(c_t, s_t, num_steps=300, alpha=1.0, beta=1e5, init_noise=True, log_every=150)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(out_c)
axes[0].set_title("init = content")
axes[0].axis("off")
axes[1].imshow(out_n)
axes[1].set_title("init = noise")
axes[1].axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/nst/init_compare.png", dpi=150, bbox_inches="tight")
plt.show()

## Часть 2 — DCGAN на EMNIST (digits)

In [ ]:
tfm = transforms.Compose(
    [
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

data_root = os.path.join(OUT, "data_emnist")
train_set = datasets.EMNIST(root=data_root, split="digits", train=True, download=True, transform=tfm)
loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))

xb, _ = next(iter(loader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("EMNIST digits [−1, 1]")
plt.imshow(
    np.transpose(
        make_grid(xb[:64], nrow=8, normalize=True, value_range=(-1, 1)).cpu().numpy(), (1, 2, 0)
    )
)
plt.show()

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(img_channels, 64, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(64, 128, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(128, 256, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(256, 1, 4, 1, 0, bias=False)),
        )

    def forward(self, img):
        return self.net(img).view(-1)


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


latent_dim = 100
G = Generator(latent_dim=latent_dim, img_channels=1).to(device)
D = Discriminator(img_channels=1).to(device)
G.apply(weights_init)
D.apply(weights_init)

criterion = nn.BCEWithLogitsLoss()
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

fixed_noise = torch.randn(16, latent_dim, 1, 1, device=device)
print("G params:", sum(p.numel() for p in G.parameters()))
print("D params:", sum(p.numel() for p in D.parameters()))

In [ ]:
num_epochs = 50
snap_epochs = [1, 5, 10, 20, 50]
history = {"loss_D": [], "loss_G": [], "D_x": [], "D_G_z": []}
saved_grids = {}

for epoch in range(num_epochs):
    eld = elg = edx = edgz = 0.0
    for real_images, _ in loader:
        bs = real_images.size(0)
        real_images = real_images.to(device)
        real_labels = torch.full((bs,), 0.9, device=device)
        fake_labels = torch.zeros(bs, device=device)

        z = torch.randn(bs, latent_dim, 1, 1, device=device)
        fake = G(z)
        d_real = D(real_images)
        d_fake = D(fake.detach())
        loss_D = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        z = torch.randn(bs, latent_dim, 1, 1, device=device)
        fake = G(z)
        d_g = D(fake)
        loss_G = criterion(d_g, torch.ones(bs, device=device))
        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        eld += loss_D.item()
        elg += loss_G.item()
        edx += torch.sigmoid(d_real).mean().item()
        edgz += torch.sigmoid(d_g).mean().item()

    n = len(loader)
    history["loss_D"].append(eld / n)
    history["loss_G"].append(elg / n)
    history["D_x"].append(edx / n)
    history["D_G_z"].append(edgz / n)

    print(
        f"Epoch [{epoch+1:02d}/{num_epochs}] loss_D={history['loss_D'][-1]:.4f} "
        f"loss_G={history['loss_G'][-1]:.4f} D(x)={history['D_x'][-1]:.3f} D(G(z))={history['D_G_z'][-1]:.3f}"
    )

    ep = epoch + 1
    if ep in snap_epochs:
        with torch.no_grad():
            fakes = G(fixed_noise)
            saved_grids[ep] = make_grid(fakes, nrow=4, normalize=True, value_range=(-1, 1)).cpu()
            save_image(fakes, f"{OUT}/gan/epoch_{ep:03d}.png", nrow=4, normalize=True, value_range=(-1, 1))

torch.save({"G": G.state_dict(), "D": D.state_dict(), "history": history}, f"{OUT}/gan/dcgan_emnist.pth")
print("Сохранено:", f"{OUT}/gan/dcgan_emnist.pth")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["loss_D"], label="loss_D")
axes[0].plot(history["loss_G"], label="loss_G")
axes[0].legend()
axes[0].set_title("Потери")
axes[0].grid(True, alpha=0.3)
axes[1].plot(history["D_x"], label="D(x)")
axes[1].plot(history["D_G_z"], label="D(G(z))")
axes[1].axhline(0.5, color="k", ls="--", alpha=0.4)
axes[1].legend()
axes[1].set_title("Уверенность D")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/gan/curves.png", dpi=150)
plt.show()

order = [e for e in snap_epochs if e in saved_grids]
if order:
    fig, axes = plt.subplots(1, len(order), figsize=(4 * len(order), 4))
    for ax, ep in zip(np.atleast_1d(axes), order):
        g = saved_grids[ep].numpy()
        ax.imshow(np.transpose(g, (1, 2, 0)))
        ax.set_title(f"Эпоха {ep}")
        ax.axis("off")
    plt.suptitle("Fixed noise: эволюция")
    plt.tight_layout()
    plt.savefig(f"{OUT}/gan/evolution_epochs.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
def denorm_gan(t):
    return (t * 0.5 + 0.5).clamp(0, 1)


G.eval()
with torch.no_grad():
    big = G(torch.randn(64, latent_dim, 1, 1, device=device))
save_image(big, f"{OUT}/gan/final_grid.png", nrow=8, normalize=True, value_range=(-1, 1))

plt.figure(figsize=(10, 10))
plt.axis("off")
plt.title("64 сгенерированных изображения")
plt.imshow(
    np.transpose(
        make_grid(big.cpu(), nrow=8, normalize=True, value_range=(-1, 1)).numpy(), (1, 2, 0)
    )
)
plt.savefig(f"{OUT}/gan/final_grid_show.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance

NUM_FID_SAMPLES = 10_000

fid = FrechetInceptionDistance(feature=2048).to(device)


def prepare_for_fid(images):
    """[N,1,32,32] в [-1,1] → uint8 [N,3,75,75] для Inception."""
    x = denorm_gan(images)
    x = x.repeat(1, 3, 1, 1)
    x = F.interpolate(x, size=(75, 75), mode="bilinear", align_corners=False)
    return (x * 255).byte()


cnt = 0
for real_images, _ in loader:
    real_images = real_images.to(device)
    fid.update(prepare_for_fid(real_images), real=True)
    cnt += real_images.size(0)
    if cnt >= NUM_FID_SAMPLES:
        break

G.eval()
cnt = 0
with torch.no_grad():
    while cnt < NUM_FID_SAMPLES:
        z = torch.randn(128, latent_dim, 1, 1, device=device)
        fakes = G(z)
        fid.update(prepare_for_fid(fakes), real=False)
        cnt += fakes.size(0)

score = fid.compute()
print(f"FID (n={NUM_FID_SAMPLES}): {score.item():.2f}")

In [ ]:
z1 = torch.randn(1, latent_dim, 1, 1, device=device)
z2 = torch.randn(1, latent_dim, 1, 1, device=device)
alphas = torch.linspace(0, 1, steps=10)

G.eval()
with torch.no_grad():
    interp = torch.cat([(1 - a) * z1 + a * z2 for a in alphas], dim=0)
    imgs = G(interp)

plt.figure(figsize=(14, 2))
plt.imshow(
    np.transpose(
        make_grid(imgs.cpu(), nrow=10, normalize=True, value_range=(-1, 1)).numpy(), (1, 2, 0)
    )
)
plt.axis("off")
plt.title("Интерполяция в Z")
plt.tight_layout()
plt.savefig(f"{OUT}/gan/interpolation.png", dpi=150, bbox_inches="tight")
plt.show()